In [ ]:
!nvidia-smi

Fri Aug 28 16:59:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U unsloth unsloth_zoo
!pip install -U datasets trl transformers accelerate bitsandbytes



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/2

In [ ]:
dataset = load_dataset("lavita/MedQuAD", split="train")

print(dataset)
print("Number of records:", len(dataset))
print(dataset.column_names)


README.md:   0%|          | 0.00/2.77k [00:00<?, ?B/s]

data/train-00000-of-00001-e36383d177026d(…): reconstructing file:   0%|          |  0.00B / 10.7MB            

data/train-00000-of-00001-e36383d177026d(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

Dataset({
    features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
    num_rows: 47441
})
Number of records: 47441
['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer']


In [ ]:
print(dataset[0]["question"])
print()
print(dataset[0]["answer"])

What is (are) keratoderma with woolly hair ?

Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appear until adolescence or later. Complications of cardiomyopathy can include an abnormal heartbeat (arrhy

In [ ]:
dataset = dataset.shuffle(seed=42)

train_dataset = dataset.select(range(1500))
eval_dataset = dataset.select(range(1500, 1600))

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))

Training examples: 1500
Evaluation examples: 100


In [ ]:
def format_example(example):
    return {
        "text": (
            "<|user|>\n"
            f"{example['question']}\n"
            "<|assistant|>\n"
            f"{example['answer']}"
        )
    }

train_dataset = train_dataset.map(format_example)
eval_dataset = eval_dataset.map(format_example)

print(train_dataset[0]["text"])

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

<|user|>
How to prevent Marburg hemorrhagic fever (Marburg HF) ?
<|assistant|>
Preventive measures against Marburg virus infection are not well defined, as transmission from wildlife to humans remains an area of ongoing research. However, avoiding fruit bats, and sick non-human primates in central Africa, is one way to protect against infection. 
 
Measures for prevention of secondary, or person-to-person, transmission are similar to those used for other hemorrhagic fevers. If a patient is either suspected or confirmed to have Marburg hemorrhagic fever, barrier nursing techniques should be used to prevent direct physical contact with the patient. These precautions include wearing of protective gowns, gloves, and masks; placing the infected individual in strict isolation; and sterilization or proper disposal of needles, equipment, and patient excretions. 
 
In conjunction with the World Health Organization, CDC has developed practical, hospital-based guidelines, titled: Infection Contro

In [ ]:
max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    full_finetuning=False,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.8.22 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="medical_qlora_output",
        report_to="none",
        fp16=True,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,500 | Num Epochs = 1 | Total steps = 375
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.371395
20,1.623783
30,1.543413
40,1.245399
50,1.379442
60,1.275501
70,1.238224
80,1.088773
90,1.247923
100,1.075834


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in medical_qlora_output/checkpoint-375/tokenizer_config.json.


In [ ]:
print(trainer_stats)

TrainOutput(global_step=375, training_loss=1.1615565999348958, metrics={'train_runtime': 717.5313, 'train_samples_per_second': 2.091, 'train_steps_per_second': 0.523, 'total_flos': 1310805378376704.0, 'train_loss': 1.1615565999348958, 'epoch': 1.0})


In [ ]:
print(trainer.state.log_history[-5:])

[{'loss': 1.194065475463867, 'grad_norm': 0.9433937072753906, 'learning_rate': 2e-05, 'epoch': 0.9066666666666666, 'step': 340}, {'loss': 0.8237175941467285, 'grad_norm': 0.49095600843429565, 'learning_rate': 1.4594594594594596e-05, 'epoch': 0.9333333333333333, 'step': 350}, {'loss': 1.013011360168457, 'grad_norm': 1.0827082395553589, 'learning_rate': 9.18918918918919e-06, 'epoch': 0.96, 'step': 360}, {'loss': 1.108686351776123, 'grad_norm': 1.3567583560943604, 'learning_rate': 3.783783783783784e-06, 'epoch': 0.9866666666666667, 'step': 370}, {'train_runtime': 717.5313, 'train_samples_per_second': 2.091, 'train_steps_per_second': 0.523, 'total_flos': 1310805378376704.0, 'train_loss': 1.1615565999348958, 'epoch': 1.0, 'step': 375}]


In [ ]:
def ask_medical_question(question):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
    )

    answer = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    return answer

In [ ]:
question = "What is anemia?"
answer = ask_medical_question(question)

print("Question:", question)
print()
print("Model Answer:", answer)

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is anemia?

Model Answer: Anemia is a condition in which the number of red blood cells or hemoglobin levels fall below normal. Red blood cells contain the protein called hemoglobin, which carries oxygen throughout the body. Anemia occurs when there aren't enough healthy red blood cells to carry adequate amounts of oxygen to tissues and organs. There are many types of anemia. The most common types include iron deficiency anemia, folate deficiency anemia, vitamin B12 deficiency anemia, and sickle cell disease. Anemia can be treated with iron supplements, medications that increase red blood cell production (erythropoietin), or transfusions of red blood cells. In some cases, surgery may be needed to correct the underlying cause of the anemia.


In [ ]:
questions = [
    "What is hypertension?",
    "What are common symptoms of asthma?",
    "Why is hand hygiene important?",
]

for q in questions:
    print("=" * 80)
    print("QUESTION:", q)
    print("ANSWER:", ask_medical_question(q))

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: What is hypertension?


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: None of the following statements about hypertension is correct. Hypertension is defined as blood pressure readings that consistently exceed 140/90 mmHg on at least three separate occasions (i.e., when measured in two or more different locations). Hypertension is diagnosed based on these criteria and does not require further testing to confirm the diagnosis. If you have high blood pressure, your doctor may want to measure your blood pressure at other times during the day. They may also do tests for underlying conditions that can cause hypertension such as kidney disease, sleep apnea, and diabetes. Your doctor will also check how well your heart functions with an echocardiogram. If your doctor thinks there might be a problem with your kidneys, they may
QUESTION: What are common symptoms of asthma?


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your question. None of the above answers your
QUESTION: Why is hand hygiene important?
ANSWER: None of us want to get sick from germs that we touch on our hands. We need to wash our hands with soap and water for 20 seconds especially after going to the bathroom, before

In [ ]:
for i in range(5):
    question = eval_dataset[i]["question"]
    reference = eval_dataset[i]["answer"]

    prediction = ask_medical_question(question)

    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("\nREFERENCE ANSWER:")
    print(reference[:500])
    print("\nMODEL ANSWER:")
    print(prediction)

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
How many people are affected by Unverricht-Lundborg disease ?

REFERENCE ANSWER:
Progressive myoclonus epilepsy is a rare condition. Unverricht-Lundborg disease is believed to be the most common cause of this type of epilepsy, but its worldwide prevalence is unknown. Unverricht-Lundborg disease occurs most frequently in Finland, where approximately 4 in 100,000 people are affected.

MODEL ANSWER:
The prevalence of ULD is unknown; however, it has been estimated to affect 1 in every 250,000 people worldwide. The condition affects males and females equally. It typically begins between ages 30 and 40 years, but can occur at any age. The exact cause of the disorder is unknown. In most cases, no specific cause can be identified. ULD appears to have an autosomal dominant inheritance pattern (meaning that one copy of the mutated gene in each cell is sufficient to cause the disorder). This means that individuals with only one copy of the mutated gene will develop symptoms. However, so

TypeError: 'NoneType' object is not subscriptable

In [ ]:
adapter_path = "medical_qlora_adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("Adapter saved to:", adapter_path)

Unsloth: Restored added_tokens_decoder metadata in medical_qlora_adapter/tokenizer_config.json.


Adapter saved to: medical_qlora_adapter


In [ ]:
import os

print(os.listdir(adapter_path))

['adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json', 'chat_template.jinja', 'README.md', 'tokenizer.json']
